# Tutorial: Layer 1 SSG Hidden Weakness Mining


## 1. 목적

            이 노트북은 1번 레이어인 **SSG 숨은 약점 마이닝**을 학부생에게 설명하기 위한 자료다.

            핵심 질문은 단순히 "SSG가 못하는 게 무엇인가?"가 아니다.

            > SSG에서만 특이하게 나타나는 병목이 무엇이고, 그 병목을 외국인/아시아쿼터 후보의 어떤 능력으로 번역할 수 있는가?

            ## 사용한 모델/방법

            - 단순 기준선: OPS, ERA, 득점, 실점 같은 일반 지표로 먼저 설명 가능한지 확인
            - z-score composite: 팀/역할/상황별 순위 차이를 표준화해서 숨은 약점 점수화
            - anomaly mining: 다른 팀과 비교했을 때 SSG만 튀는 상황을 탐색
            - text corroboration: 기사/인터뷰 메타데이터로 현장 언어가 같은 방향을 가리키는지 확인
            - feature contract: 메시지를 후보 평가에 연결 가능한 변수 목록으로 고정


## 2. 오늘 배울 흐름

            1. 전체 6개 레이어 중 1번의 현재 상태를 확인한다.
            2. 모델 청사진에서 1번에 해당하는 모델만 뽑아본다.
            3. 메시지 후보와 증거 카드를 확인한다.
            4. 메시지가 후보 평가 변수로 번역됐는지 확인한다.


In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = ROOT.parent

OUT = ROOT / "outputs" / "tables"

def read_table(filename):
    path = OUT / filename
    if not path.exists():
        print(f"[missing] {path}")
        return pd.DataFrame()
    return pd.read_csv(path)

def take_cols(df, cols, n=10):
    keep = [c for c in cols if c in df.columns]
    if not keep:
        return df.head(n)
    return df.loc[:, keep].head(n)

def count_by(df, cols):
    keep = [c for c in cols if c in df.columns]
    if not keep or df.empty:
        return pd.DataFrame()
    return df.groupby(keep, dropna=False).size().reset_index(name="rows").sort_values("rows", ascending=False)

def assert_candidate_names_locked(df):
    sensitive_cols = {"player_name", "search_name", "team_or_org", "player_id"}
    exposed = sensitive_cols.intersection(df.columns)
    if exposed:
        print(f"Candidate-sensitive columns exist but are not displayed here: {sorted(exposed)}")


In [ ]:
gate = read_table("recruitment_gate_status_v33.csv")
take_cols(gate, ["gate", "layer", "progress_pct", "status", "decision", "blocking_gap"], n=6)


In [ ]:
blueprint = read_table("modeling_blueprint_registry_v1.csv")
layer1_models = blueprint[
    blueprint["layer"].astype(str).str.contains("ssg_need|text_mining|gate", case=False, regex=True, na=False)
]
take_cols(layer1_models, ["model_id", "model_family", "target_or_score", "main_features", "validation", "promotion_rule"])


## 3. 메시지 마이닝 결과

            아래 표는 후보 이름이 아니라 **슬롯별 메시지**를 점수화한 결과다.

            해석 포인트:

            - evidence_score가 높을수록 데이터 카드가 많이 붙었다.
            - novelty는 뻔하지 않은 정도다.
            - SSG specificity는 다른 팀에도 통하는 말이 아니라 SSG에 특화된 정도다.
            - weakest_caveat는 발표에서 먼저 방어해야 할 약점이다.


In [ ]:
slot_scores = read_table("message_mining_slot_scores_v1.csv")
take_cols(
    slot_scores.sort_values("total_evidence_score", ascending=False),
    ["slot", "message_id", "message", "cards", "total_evidence_score", "avg_novelty", "avg_ssg_specificity", "avg_actionability", "weakest_caveat"],
    n=10,
)


In [ ]:
cards = read_table("message_mining_evidence_cards_v1.csv")
evidence_summary = (
    cards.groupby("slot", dropna=False)
    .agg(
        cards=("card_id", "count"),
        total_evidence=("evidence_score", "sum"),
        avg_evidence=("evidence_score", "mean"),
        avg_novelty=("novelty_1_5", "mean"),
        avg_ssg_specificity=("ssg_specificity_1_5", "mean"),
    )
    .reset_index()
    .sort_values("total_evidence", ascending=False)
)
evidence_summary


## 4. 메시지를 후보 변수로 바꾸기

            데이터 마이닝에서 가장 중요한 단계는 "멋진 문장"을 "측정 가능한 변수"로 바꾸는 것이다.

            예를 들어 "ABS-native load-bearing starter"는 다음처럼 바뀐다.

            - workload proxy: 최근 80-90구 이상 던진 경기 경험
            - command proxy: 3-ball 비율, BB+HBP%, first-pitch non-ball
            - damage control proxy: HR%, hard-hit%, barrel%, RISP/on-base damage


In [ ]:
audit = read_table("layer1_candidate_feature_join_audit_v0_1.csv")
take_cols(
    audit,
    ["slot", "message_feature", "candidate_proxy_column", "candidate_proxy_status", "join_status", "blocking_gap"],
    n=12,
)


In [ ]:
closure = read_table("layer1_freeze_closure_matrix_v0_1.csv")
take_cols(closure, closure.columns.tolist(), n=12)


## 5. 발표용 한 줄

            1번 레이어는 결론을 미리 정한 것이 아니라, SSG의 상황별 병목을 찾고 그 병목을 후보 평가 변수로 번역한 단계다.

            **핵심 메시지:** SSG의 약점은 단순 공격/수비 총량이 아니라, 경기 흐름의 특정 구간에서 끊기는 구조다. 그래서 후보도 일반적인 좋은 선수가 아니라 그 구조를 복구하는 능력을 기준으로 봐야 한다.

            ## 연습문제

            `message_mining_evidence_cards_v1.csv`에서 evidence_score가 높은 카드 3개를 골라, 각 카드가 어떤 후보 평가 변수로 번역될 수 있는지 한 줄씩 적어보자.


In [ ]:
top_cards = cards.sort_values("evidence_score", ascending=False).head(3)
take_cols(top_cards, ["card_id", "slot", "theme", "claim", "metric", "observed", "comparison", "evidence_score", "caveat"], n=3)
